In [3]:
import ollama

def ask_question_local_llm(prompt):
    # print(f"User asked: {prompt}")
    # my_client.chat.completions.create

    # Run a prompt against a local model (e.g., llama2)
    response = ollama.chat(
        model='llama3',
        messages=[
            {"role": "system", "content": "You are a helpful AI assitant - Respond in one line"},
            {"role": "user", "content": prompt}
        ]
    )
    return response['message']['content']

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")

In [6]:
my_client = OpenAI(api_key=my_api_key)
# my_client

def ask_question_open_gpt_41(prompt):

    # print(f"User asked: {prompt}")
    # my_client.chat.completions.create

    llm_response = my_client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
            {"role": "user", "content": prompt}
        ]
    )
    return llm_response.choices[0].message.content  


def ask_question_open_gpt_5(prompt):

    # print(f"User asked: {prompt}")
    # my_client.chat.completions.create

    llm_response = my_client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
            {"role": "user", "content": prompt}
        ]
    )
    return llm_response.choices[0].message.content

In [7]:
prompt = "What is the capital of France?"

response_local_llm = ask_question_local_llm(prompt)
response_open_ai_41 = ask_question_open_gpt_41(prompt)
response_open_ai_5 = ask_question_open_gpt_5(prompt)

print(f"Response from local LLM: {response_local_llm}")
print(f"Response from OpenAI GPT-4.1: {response_open_ai_41}")
print(f"Response from OpenAI GPT-5: {response_open_ai_5}")

Response from local LLM: The capital of France is Paris.
Response from OpenAI GPT-4.1: The capital of France is Paris.
Response from OpenAI GPT-5: Paris.


In [15]:
import time
while True:
    # Ask user for a question
    user_prompt = input("Ask something: ")
    

    if (user_prompt.lower() != 'quit'):
        # Get and print the response
        
        print ("\n\n-------------------LOCAL LLM RESPONSE-------------------")
        start = time.time()
        response_local = ask_question_local_llm(user_prompt)
        end = time.time() 
        time_taken_local=end-start 
        print(f"Time taken by Local LLM: {end - start} seconds")
        print("\nLocal LLM says:", response_local)     

        print ("-------------------OPEN AI RESPONSE - GPT-4.1 -------------------")
        start = time.time()
        response_openai = ask_question_open_gpt_41(user_prompt)
        end = time.time() 
        time_taken_openai_41=end-start   
        print(f"Time taken by OpenAI: {end - start} seconds")
        print("\nOpenAI says:", response_openai)

        print ("\n\n-------------------OPEN AI RESPONSE - GPT-5 -------------------")
        start = time.time()
        response_openai_5 = ask_question_open_gpt_5(user_prompt)
        end = time.time()
        time_taken_openai_5=end-start
        print(f"Time taken by OpenAI GPT-5: {end - start} seconds")
        print("\nOpenAI GPT-5 says:", response_openai_5)

           

        # add delay of 3 seconds
        time.sleep(3)
    else:
        print("Exiting...")
        break    



-------------------LOCAL LLM RESPONSE-------------------
Time taken by Local LLM: 29.664854049682617 seconds

Local LLM says: Here is the required output:

**SECTION 1 — Executive Summary**
A 23% decrease in inbound shipment processing was observed across three West Coast warehouse locations, affecting Oakland, Reno, and Phoenix, approximately 14 hours after a new inventory synchronization service was deployed. The issue is characterized by API timeout spikes, duplicate inventory records, delayed barcode scan propagation, and increased message queue lag.

**SECTION 2 — Root Cause Analysis**

* **Primary Cause**: Incomplete or inconsistent data processing within the new inventory synchronization service.
	+ Evidence: API timeout spikes, Redis cache hit rate drop, PostgreSQL write throughput increase, and Message queue lag.
* **Secondary Causes**:
	+ Duplicate inventory records in Reno (12% of active SKUs).
	+ Delayed barcode scan propagation in Phoenix.
	+ Possible race condition duri

In [19]:
def compare_responses():
    comparison_response = my_client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": '''You are an evaluator. Please evaluate the quality of the response by different models.
            Put as a JSON response with keys as model names and values as score out of 10. Also add a justification for the score.
             time taken should be of 80% of weightage and rest is 20%'''},
            {"role": "user", "content": "Local LLM response: " + response_local_llm + "\n time taken by local llm :" + str(time_taken_local)
            +"\n\nOpenAI GPT-4.1 response: " + response_openai + "\n time taken by OpenAI GPT-4.1  :" + str(time_taken_openai_41)
            + "\n\nOpenAI GPT-5 response: " + response_openai_5+ "\n time taken by penAI GPT-5 :" + str(time_taken_openai_5)
            }
        ]
    )
    return comparison_response.choices[0].message.content


In [20]:
comparison= compare_responses()
print("\n\n comparison of repsones:\n", comparison)



 comparison of repsones:
 {
  "Local LLM": {
    "score": 2.57,
    "justification": "Answer is factually correct (capital of France is Paris) but entirely unrelated to the requested evaluation task. It demonstrates minimal engagement with the incident analysis prompt, resulting in low content quality. Time was moderate, but due to poor task alignment, the overall score remains low."
  },
  "OpenAI GPT-4.1": {
    "score": 9.80,
    "justification": "Provides a detailed, structured incident report with clear sections (Executive Summary, Root Cause, Risk, Actions, etc.), including a structured JSON section and actionable recommendations. Fast response relative to GPT-5 and comprehensive coverage contribute to a high quality score. Time efficiency also supports the high overall score."
  },
  "OpenAI GPT-5": {
    "score": 1.90,
    "justification": "Contents are thorough and well-structured with extensive analysis and concrete next steps. However, the response is slower (longer time),